# 02 – Sessions laden und speichern

Dieses Notebook extrahiert alle **Nutzer-Queries (Sessions)** aus dem LongEval-SCI-Datensatz
und speichert sie als strukturierte CSV-Datei für die weiteren Pipeline-Schritte.

Da der Testdatensatz einzelne Queries (keine echten Mehrschritt-Sessions) enthält,
wird jede Query als eigene Session mit `num_queries=1` behandelt.

**Voraussetzung:** `00_Data_loading.ipynb` wurde erfolgreich ausgeführt.

**Ausgabe:** `data/sessions.csv` und `data/sessions_first_all.csv`

## 1. Queries laden und als Sessions strukturieren

Iteriert über alle Queries des Datensatzes und baut eine flache Session-Struktur auf.
Felder: `session_id`, `num_queries`, `queries`, `first_query`, `all_queries`.

In [3]:
from ir_datasets_longeval import load
import pandas as pd
import os

# Snapshot-3 laden (greift auf den lokalen Cache zurück)
dataset = load("longeval-sci-2026/snapshot-3")

sessions = []
for q in dataset.queries_iter():
    # Jede Query wird als eigenständige Session behandelt
    # (Testdatensatz enthält keine echten Multi-Query-Sessions)
    sessions.append({
        "session_id": q.query_id,
        "num_queries": 1,
        "queries": [q.text],
        "first_query": q.text,   # Primäre Query für Retrieval in Notebook 03
        "all_queries": q.text    # Fallback für Multi-Query-Modus
    })

print(f"Test-Queries geladen: {len(sessions)}")
print(f"Beispiel: {sessions[0]}")

Test-Queries geladen: 381
Beispiel: {'session_id': 'd19e7b4de358bcfeb89e06095dc26419', 'num_queries': 1, 'queries': ['peer-to-peer communication'], 'first_query': 'peer-to-peer communication', 'all_queries': 'peer-to-peer communication'}


## 2. Als DataFrame speichern

Konvertiert die Sessions in einen pandas DataFrame und exportiert ihn als CSV.
Beide Dateien sind inhaltlich identisch und decken verschiedene Downstream-Nutzungen ab.

In [4]:
sessions_df = pd.DataFrame(sessions)

print("Anzahl Sessions:", len(sessions_df))

# Ausgabeverzeichnis anlegen, falls noch nicht vorhanden
os.makedirs("data", exist_ok=True)

# Zwei Kopien für verschiedene Downstream-Schritte
sessions_df.to_csv("data/sessions.csv", index=False)
sessions_df.to_csv("data/sessions_first_all.csv", index=False)
print("Gespeichert: data/sessions.csv")

Anzahl Sessions: 381
Gespeichert: data/sessions.csv
